# ClauseDraft — hosted QLoRA run


The runtime disk is wiped when the session ends. Checkpoints and the finished adapter are written to Google Drive, so a dropped session can be continued.

This notebook does not install `requirements.txt`. That file pins a CUDA 13.0 torch wheel for an RTX 50-series laptop. Colab already has a CUDA torch, and replacing it breaks the GPU.

In [ ]:
import shutil
import torch

if shutil.which("nvidia-smi") is None or not torch.cuda.is_available():
    raise SystemExit("No GPU in this runtime. Change the runtime type to a T4 and run this cell again.")
print(torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess
from pathlib import Path

root = Path("/content/domain-specific-llm-fine-tuning")
if not (root / ".git").is_dir():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Armiin-h/domain-specific-llm-fine-tuning.git",
        str(root),
    ])
else:
    subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only"])
os.chdir(root)
print(root)

In [ ]:
import subprocess
import sys
import torch

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
import torch as torch_after
if not torch_after.cuda.is_available():
    raise SystemExit("The install replaced torch with a CPU build. Restart the runtime and run the cells again.")
print("torch", torch.__version__, "->", torch_after.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
adapter_dir = "/content/drive/MyDrive/clausedraft-adapter"
print(adapter_dir)

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "scripts/prepare_corpus.py"])
subprocess.check_call([sys.executable, "scripts/build_jsonl.py"])

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

root = Path("/content/domain-specific-llm-fine-tuning")
train_file = root / "data" / "processed" / "train.jsonl"
local_out = Path("/content/clausedraft-adapter")
drive_out = Path("/content/drive/MyDrive/clausedraft-adapter")
print("train file:", train_file, "found" if train_file.is_file() else "MISSING")
if not train_file.is_file():
    raise SystemExit("Train file is missing. Rerun the prepare cell.")
print(f"free space on /content: {shutil.disk_usage('/content').free / 1024**3:.1f} GB")
print("gpu:", torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())
local_out.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_CACHE"] = "/content/hf-hub"
code = subprocess.call(
    [sys.executable, "-u", "scripts/train.py", "--output-dir", str(local_out)],
    cwd=root,
    env=env,
)
print("train exit code:", code)
if code != 0:
    raise SystemExit(code)
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_out, drive_out, dirs_exist_ok=True)
print("copied adapter to", drive_out)

Run the next cell only if this session died after a checkpoint. It copies the Drive folder back onto the Colab disk and continues. A finished run already copied `adapter_model.safetensors`, `loss.svg`, and `model_card.md` to Drive. Do not run it after a successful train cell.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

root = Path("/content/domain-specific-llm-fine-tuning")
local_out = Path("/content/clausedraft-adapter")
drive_out = Path("/content/drive/MyDrive/clausedraft-adapter")
if drive_out.is_dir():
    local_out.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_out, local_out, dirs_exist_ok=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_CACHE"] = "/content/hf-hub"
code = subprocess.call(
    [sys.executable, "-u", "scripts/train.py", "--resume", "--output-dir", str(local_out)],
    cwd=root,
    env=env,
)
print("resume exit code:", code)
if code != 0:
    raise SystemExit(code)
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_out, drive_out, dirs_exist_ok=True)
print("copied adapter to", drive_out)